# AI-RVC 一键 AI 翻唱 (Colab)

[Open in Colab](https://colab.research.google.com/github/mason369/AI-RVC/blob/master/AI_RVC_Colab.ipynb)

AI-RVC 的 Google Colab 运行入口。Notebook 会创建独立 Python 3.10 环境并安装全部项目依赖，保证 `fairseq==0.12.2` 与 `audio-separator==0.47.0` 稳定运行。

### 默认处理链路模型

| 环节 | 默认模型 |
|------|----------|
| 人声 / 纯伴奏分离 | `hybrid:leap_xe90_vocals+leap62_instrumental`：Leap XE 90 提取人声，BS-RoFormer Leap Instrumental 62 提取纯伴奏 |
| 主唱 / 和声分离 | [MVSep 9205](https://www.mvsep.com/quality_checker/entry/9205) (`ensemble:mvsep_9205_avg`)：三模型 `avg_wave` 提取主唱与带和声伴奏，差分导出纯和声 |
| 去混响 | [dereverb_bs_roformer_anvuew_sdr_22.5050.ckpt](https://huggingface.co/anvuew/dereverb_bs_roformer) |
| 内容特征 | [Transformers HuBERT Base](https://huggingface.co/lj1995/VoiceConversionWebUI/tree/main/hubert_base) |
| F0 音高 | [rmvpe.pt](https://huggingface.co/lj1995/VoiceConversionWebUI/blob/main/rmvpe.pt) |
| 音色转换 | [RVC v1/v2](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI) 原生 256/768 维 `.pth` + 可选 [FAISS](https://github.com/facebookresearch/faiss) `.index` |

### 使用步骤

1. 在菜单栏选择 **T4 GPU** 运行时；
2. 依次运行第 1 至 5 步准备 Python 3.10 环境与基础模型；
3. 可在第 6 步查看内置声线标识，并在第 7 步上传歌曲或干声执行批量翻唱并下载成品 ZIP。

Select a T4 GPU runtime, run steps 1–5 to prepare the environment and models, then upload and convert songs in step 7 (step 6 lists available voices). See [README](https://github.com/mason369/AI-RVC) for full documentation.


## 1. 检查 Colab 运行时

这一步会显示当前 GPU 和 Colab 默认 Python。后面安装时会使用独立 Python 3.10 环境。

In [ ]:
import sys
import subprocess

print('Colab 默认 Python:', sys.version)
print('\nGPU 信息:')
subprocess.run(['nvidia-smi'], check=True)

## 2. 克隆或刷新仓库

如果 `/content/AI-RVC` 已存在，只接受无未提交修改的快进更新；有本地修改或分支分歧时停止，不清除用户代码。

In [ ]:
%%bash
set -euo pipefail
cd /content
if [ -d AI-RVC/.git ]; then
  git -C AI-RVC fetch origin master
  if [ -n "$(git -C AI-RVC status --porcelain --untracked-files=no)" ]; then
    echo '仓库有未提交修改，请先保存后再更新。' >&2
    exit 1
  fi
  git -C AI-RVC merge --ff-only origin/master
else
  git clone https://github.com/mason369/AI-RVC.git AI-RVC
fi
cd /content/AI-RVC
git rev-parse --short HEAD
ls -la

## 3. 创建 Python 3.10 环境并安装依赖

这个步骤会安装 `uv`，用它准备 Python 3.10，然后调用项目自带 `install.py --no-run`。如果任何依赖安装失败，单元格会直接停止并显示错误。

In [ ]:
%%bash
set -euo pipefail
cd /content/AI-RVC

sudo apt-get update -y
sudo apt-get install -y ffmpeg build-essential python3-dev curl git

if ! command -v uv >/dev/null 2>&1; then
  curl -LsSf https://astral.sh/uv/install.sh | sh
fi
export PATH="$HOME/.local/bin:$PATH"
uv --version
uv python install 3.10
uv venv --seed --python 3.10 venv310
PY=/content/AI-RVC/venv310/bin/python
$PY --version
$PY -m pip --version
$PY -m pip install -r pre-requirements.txt
$PY -m pip install torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu126
$PY - <<'PY'
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU PyTorch install completed but CUDA is not available.')
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
PY
uv run --python 3.10 python install.py --backend cuda --no-run

## 4. 检查依赖与默认模型配置

这里会检查关键包、Python 版本、CUDA、Colab 默认分离模型、卡拉OK模型和 DeEcho 模型是否符合当前项目配置。

In [ ]:
%%bash
set -euo pipefail
cd /content/AI-RVC
PY=/content/AI-RVC/venv310/bin/python
$PY - <<'PY'
import importlib.metadata as md
import sys
import torch
from packaging.specifiers import SpecifierSet
from infer.separator import (
    ROFORMER_DEFAULT_MODEL,
    KARAOKE_DEFAULT_MODEL,
    ROFORMER_DEREVERB_DEFAULT_MODEL,
)

assert sys.version_info[:2] == (3, 10), sys.version
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Colab runtime did not provide CUDA. Please switch runtime type to T4 GPU and rerun.')

expected = {
    'audio-separator': '==0.47.0',
    'transformers': '==4.49.0',
    'onnxruntime-gpu': '>=1.17',
    'einops': '>=0.7.0',
    'PyYAML': '>=6.0',
    'fairseq': '==0.12.2',
    'gradio': '==5.49.1',
}
for dist, constraint in expected.items():
    version = md.version(dist)
    print(f'{dist}: {version}')
    if version not in SpecifierSet(constraint):
        raise RuntimeError(f'{dist} {version} does not satisfy {constraint}')

assert ROFORMER_DEFAULT_MODEL == 'hybrid:leap_xe90_vocals+leap62_instrumental', ROFORMER_DEFAULT_MODEL
assert KARAOKE_DEFAULT_MODEL == 'ensemble:mvsep_9205_avg', KARAOKE_DEFAULT_MODEL
assert ROFORMER_DEREVERB_DEFAULT_MODEL == 'dereverb_bs_roformer_anvuew_sdr_22.5050.ckpt', ROFORMER_DEREVERB_DEFAULT_MODEL

import faiss  # noqa: F401
import onnxruntime
if 'CUDAExecutionProvider' not in onnxruntime.get_available_providers():
    raise RuntimeError('Colab ONNX Runtime did not provide CUDAExecutionProvider.')
import einops  # noqa: F401
import yaml  # noqa: F401
from audio_separator.separator import Separator  # noqa: F401
import fairseq  # noqa: F401
print('Dependency and model default checks passed.')
PY

## 5. 下载基础模型和默认分离模型

下载 [HuBERT](https://arxiv.org/abs/2106.07447)、[RMVPE](https://arxiv.org/abs/2306.15412)、[UVR5 HP2](https://huggingface.co/lj1995/VoiceConversionWebUI/blob/main/uvr5_weights/HP2_all_vocals.pth)、Leap XE vocals、Leap Instrumental pure accompaniment、MVSep 9205 子模型和 RoFormer De-Reverb，并准备内置官方 RVC 源码。模型会保存到仓库的 `assets/` 目录，固定版本官方源码会放在 `_official_rvc_runtime/<commit>/`；旧 `_official_rvc/` 不会覆盖，新版 VC 使用 Transformers HuBERT 三文件目录。

In [ ]:
%%bash
set -euo pipefail
cd /content/AI-RVC
PY=/content/AI-RVC/venv310/bin/python
$PY tools/download_models.py
$PY - <<'PY'
from tools.download_models import REQUIRED_MODELS, check_model, check_required_default_separator_models, get_missing_default_separator_model_files
missing = [name for name in REQUIRED_MODELS if not check_model(name)]
if missing:
    raise RuntimeError('Missing required models: ' + ', '.join(missing))
if not check_required_default_separator_models():
    raise RuntimeError('Missing default separator models: ' + ', '.join(get_missing_default_separator_model_files()))
print('Required models are present:', ', '.join(REQUIRED_MODELS))
print('Default separator models are present.')
PY

## 6. 可选：查看或下载声线模型

处理链路模型已经在前面准备好。这里的 `.pth` 声线模型是 RVC 目标音色，不属于分离 / F0 / HuBERT 等处理模型。可以先下载一个示例模型，也可以跳过后在 Web UI 中下载。

In [ ]:
%%bash
set -euo pipefail
cd /content/AI-RVC
PY=/content/AI-RVC/venv310/bin/python
$PY - <<'PY'
from tools.character_models import list_available_characters, list_available_series
print('Available series:')
for series in list_available_series():
    print(' -', series)
chars = list_available_characters()
print('\nFirst 20 voice models:')
for char in chars[:20]:
    print(f" - {char['name']}: {char.get('display', char.get('description', ''))}")
print('\nTotal:', len(chars))
PY

In [ ]:
# Optional: download one example voice model.
# Change character_name as needed, then run this cell.

# %%bash
# set -euo pipefail
# cd /content/AI-RVC
# PY=/content/AI-RVC/venv310/bin/python
# $PY - <<'PY'
# from tools.character_models import download_character_model
# character_name = 'rin'
# if not download_character_model(character_name):
#     raise RuntimeError(f'Failed to download voice model: {character_name}')
# print(f'Downloaded voice model: {character_name}')
# PY

## 7. 批量翻唱 / Batch covers

先多选音频，或按顺序把会话文件路径填入 `INPUT_FILES`。下面的处理单元中，整曲选 `song`，已分离的人声选 `vocals`。调整升降调或原主唱混入比例后，只需重跑处理单元。已完成的分轨可在同一会话复用。成功的 WAV 打包下载，各首分轨保存在项目的 `outputs` 目录。

Select one or more files, or enter session paths in `INPUT_FILES` in the desired order. Use `song` for mixed music or `vocals` for isolated vocals. Rerun the processing cell to change pitch or the original vocal blend and reuse completed stages in this session. Successful WAVs download in a ZIP. Individual tracks remain in the project's `outputs` directory.

In [ ]:
from google.colab import files

INPUT_FILES = []
batch_inputs = INPUT_FILES or list(files.upload(target_dir='/content/ai-rvc-inputs'))

In [ ]:
import sys
from google.colab import files
if '/content/AI-RVC' not in sys.path:
    sys.path.insert(0, '/content/AI-RVC')
from tools.notebook_batch import NotebookBatch

CHARACTER = 'hanamaru_zurakichi'
MODE = 'song'
PITCH = 0
ORIGINAL_VOCAL_MIX = None  # 0–100 %, None 使用项目设置 / use project setting

if batch_inputs:
    if 'cover_session' not in globals() or cover_session.worker.poll() is not None:
        cover_session = NotebookBatch()
    archive = cover_session.convert(batch_inputs, CHARACTER, MODE, PITCH, ORIGINAL_VOCAL_MIX)
    files.download(archive)
else:
    print('未选择文件 / No files selected.')

## 可选诊断：试跑完整处理流程 / Optional processing diagnostics

上传一段含有人声的短音乐文件到 `/content/input_audio.*`，或把自己的文件路径填到 `INPUT_AUDIO`。这个步骤会下载示例声线模型 `rin`，并依次试跑默认 hybrid 分离链路（统一解码为 44.1 kHz 双声道 Float32 WAV 后，同一份音频 分别交给 Leap XE vocals 与 Leap Instrumental pure accompaniment）、[MVSep 9205 Karaoke](https://www.mvsep.com/quality_checker/entry/9205)（三模型 `avg_wave` 处理原始整曲，输出主唱与带和声伴奏，并从 Leap 人声差分得到纯和声）、[Demucs](https://github.com/facebookresearch/demucs)、[UVR5](https://github.com/Anjok07/ultimatevocalremovergui)、官方诊断路线 等处理模式。若某个模式失败，Notebook 会停在对应步骤并显示错误信息。


In [ ]:
# Upload a short vocal music file, then set INPUT_AUDIO to its path.
# Example after upload: INPUT_AUDIO = '/content/input_audio.wav'
INPUT_AUDIO = '/content/input_audio.wav'

from pathlib import Path
if not Path(INPUT_AUDIO).exists():
    raise FileNotFoundError(
        f'Missing test audio: {INPUT_AUDIO}. Upload a short vocal music file to this path before running the matrix.'
    )
print('Matrix input:', INPUT_AUDIO, Path(INPUT_AUDIO).stat().st_size, 'bytes')


In [ ]:
%%bash
set -euo pipefail
cd /content/AI-RVC
PY=/content/AI-RVC/venv310/bin/python
$PY - <<'PY'
from tools.character_models import download_character_model
if not download_character_model('rin'):
    raise RuntimeError('Failed to download voice model: rin')
print('Voice model ready: rin')
PY


In [ ]:
import subprocess
subprocess.run([
    '/content/AI-RVC/venv310/bin/python',
    '/content/AI-RVC/tools/run_mode_matrix.py',
    '--input', str(Path(INPUT_AUDIO).resolve()),
    '--output-dir', '/content/AI-RVC/outputs/mode_matrix_colab',
    '--require-cuda',
], cwd='/content/AI-RVC', check=True)


## 8. 可选 Web 界面 / Optional web interface

第 7 步可直接在笔记本完成批量转换。仅在运行环境允许公开 Gradio 服务时，将下方开关设为 true 来启动分享链接。请查看 [Colab 使用限制](https://research.google.com/colaboratory/faq.html#limitations-and-restrictions)。

Step 7 runs batches inside the notebook. Set the switch below to true to start a public Gradio link if your runtime permits it. See the [Colab restrictions](https://research.google.com/colaboratory/faq.html#limitations-and-restrictions).

In [ ]:
%%bash
set -euo pipefail
cd /content/AI-RVC
PY=/content/AI-RVC/venv310/bin/python
START_PUBLIC_WEB_UI=false
if [ "$START_PUBLIC_WEB_UI" = true ]; then
  $PY run.py --host 0.0.0.0 --port 7860 --share
fi

## 常见问题

| 问题 | 处理方式 |
|------|----------|
| `nvidia-smi` 失败 | 在 Colab 菜单选择 GPU 运行时后重新运行 |
| `fairseq` 安装失败 | 确认第 3 步已创建并启用 Python 3.10 虚拟环境 |
| `CUDA out of memory` | 缩短单次音频时长，或在「手动调整」中关闭和声分离 |
| 模型下载失败 | 重新运行第 5 步；如遇 Hugging Face 限流可稍后再试或配置 Token |

本项目代码遵循 MIT License；模型权重与音频素材请遵守各自授权许可并合规使用。